In [1]:
import os
import json
from datetime import datetime

# =============================================
# 步骤1: 初始化 LLM
# =============================================
print("=" * 50)
print("🚀 步骤1: 初始化阿里云百炼大模型")
print("=" * 50)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)
print(f"✅ LLM 初始化完成")
print(f"   - 模型: qwen-plus")
print(f"   - API Key: {'已设置' if os.getenv('DASHSCOPE_API_KEY') else '未设置'}")

🚀 步骤1: 初始化阿里云百炼大模型
✅ LLM 初始化完成
   - 模型: qwen-plus
   - API Key: 已设置


In [2]:
# =============================================
# 步骤2: 定义工具
# =============================================
print("=" * 50)
print("🔧 步骤2: 定义可用的工具")
print("=" * 50)

from langchain_core.tools import tool

@tool(description="规划行车路线")
def get_route_plan(origin_city: str, target_city: str):
    """规划行车路线

    Args:
        origin_city: 出发城市
        target_city: 目标城市
    """
    result = f"从城市 {origin_city} 出发，到目标城市 {target_city},使用意念传送，只需要三分钟即可到达。"
    print(f"   [工具执行] get_route_plan 被调用!")
    print(f"   [工具执行] 输入参数: origin_city={origin_city}, target_city={target_city}")
    print(f"   [工具执行] 返回结果: {result}")
    return result

# 工具容器 (用于根据名称查找工具)
all_tools = {"get_route_plan": get_route_plan}

print(f"✅ 工具定义完成")
print(f"   - get_route_plan: 规划行车路线")
print(f"   - 可用工具数量: {len(all_tools)}")

🔧 步骤2: 定义可用的工具
✅ 工具定义完成
   - get_route_plan: 规划行车路线
   - 可用工具数量: 1


In [4]:
# =============================================
# 步骤3: 绑定工具到 LLM
# =============================================
print("=" * 50)
print("🔗 步骤3: 将工具绑定到 LLM")
print("=" * 50)

# 将工具绑定到 LLM，这样 LLM 可以决定何时调用工具
llm_with_tools = llm.bind_tools([get_route_plan])

print(f"✅ 工具绑定完成")
print(f"   - 当 LLM 认为需要工具时会返回 tool_calls")

🔗 步骤3: 将工具绑定到 LLM
✅ 工具绑定完成
   - 当 LLM 认为需要工具时会返回 tool_calls


In [5]:
# =============================================
# 步骤4: 构建用户输入
# =============================================
print("=" * 50)
print("👤 步骤4: 用户输入")
print("=" * 50)

# 用户的问题
query = "帮我规划一条从长沙到桂林的自驾路线"
messages = [("human", query)]

print(f"❓ 用户问题: {query}")
print(f"📝 消息历史: {messages}")

👤 步骤4: 用户输入
❓ 用户问题: 帮我规划一条从长沙到桂林的自驾路线
📝 消息历史: [('human', '帮我规划一条从长沙到桂林的自驾路线')]


In [6]:
# =============================================
# 步骤5: 第一次调用 LLM (判断是否需要工具)
# =============================================
print("=" * 50)
print("🤖 步骤5: 第一次调用 LLM")
print("=" * 50)

print(f"📤 发送请求到 LLM...")
ai_msg = llm_with_tools.invoke(messages)

print(f"\n📥 LLM 返回结果:")
print(f"   - 类型: {type(ai_msg)}")
print(f"   - content: {ai_msg.content}")
print(f"   - tool_calls: {ai_msg.tool_calls}")

# 将 LLM 的响应添加到消息历史
messages.append(ai_msg)
print(f"\n📝 更新后的消息历史长度: {len(messages)}")

🤖 步骤5: 第一次调用 LLM
📤 发送请求到 LLM...

📥 LLM 返回结果:
   - 类型: <class 'langchain_core.messages.ai.AIMessage'>
   - content: 
   - tool_calls: [{'name': 'get_route_plan', 'args': {'origin_city': '长沙', 'target_city': '桂林'}, 'id': 'call_9d10cb7949674de090d59f', 'type': 'tool_call'}]

📝 更新后的消息历史长度: 2


In [7]:
# =============================================
# 步骤6: 检查是否需要调用工具
# =============================================
print("=" * 50)
print("🔍 步骤6: 检查是否需要调用工具")
print("=" * 50)

if ai_msg.tool_calls:
    print(f"✅ 检测到需要调用 {len(ai_msg.tool_calls)} 个工具")
    for i, tool_call in enumerate(ai_msg.tool_calls, 1):
        print(f"\n   工具 #{i}:")
        print(f"   - 名称: {tool_call['name']}")
        print(f"   - 参数: {json.dumps(tool_call['args'], ensure_ascii=False)}")
else:
    print("❌ LLM 不需要调用任何工具，直接返回了答案")

🔍 步骤6: 检查是否需要调用工具
✅ 检测到需要调用 1 个工具

   工具 #1:
   - 名称: get_route_plan
   - 参数: {"origin_city": "长沙", "target_city": "桂林"}


In [13]:
# =============================================
# 步骤7: 执行工具调用
# =============================================
print("=" * 50)
print("⚙️ 步骤7: 执行工具调用")
print("=" * 50)

if ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        print(f"tool_call-----: {tool_call}")
        # 根据工具名称查找对应的工具
        tool_name = tool_call["name"].lower()
        print(f"\n🔧 查找工具: {tool_name}")
        # 判断ai获取到的tool的name是否是我们定义的tools数组，避免ai随便捏造出一个函数
        if tool_name in all_tools:
            # langChain 会把这个普通的 Python 函数包装成一个 StructuredTool 对象。
            selected_tool = all_tools[tool_name]
            print(f"all_tools的类型: {type(all_tools)}")
            print(f"selected_tool的类型: {type(selected_tool)}")
            print(f"✅ 找到工具: {selected_tool.name}")
            
            # 调用工具并获取结果
            print(f"\n⚡ 调用工具...")
            tool_msg = selected_tool.invoke(tool_call)
            
            print(f"\n📤 工具返回结果:")
            print(f"   - 类型: {type(tool_msg)}")
            print(f"   - content: {tool_msg.content}")
            
            # 将工具结果添加到消息历史
            messages.append(tool_msg)
            print(f"\n✅ 工具执行完成，消息历史已更新")
        else:
            print(f"❌ 未找到工具: {tool_name}")
else:
    print("⏭️ 没有工具需要执行")

print(f"\n📝 当前消息历史长度: {len(messages)}")

⚙️ 步骤7: 执行工具调用
tool_call-----: {'name': 'get_route_plan', 'args': {'origin_city': '长沙', 'target_city': '桂林'}, 'id': 'call_9d10cb7949674de090d59f', 'type': 'tool_call'}

🔧 查找工具: get_route_plan
all_tools的类型: <class 'dict'>
selected_tool的类型: <class 'langchain_core.tools.structured.StructuredTool'>
✅ 找到工具: get_route_plan

⚡ 调用工具...
   [工具执行] get_route_plan 被调用!
   [工具执行] 输入参数: origin_city=长沙, target_city=桂林
   [工具执行] 返回结果: 从城市 长沙 出发，到目标城市 桂林,使用意念传送，只需要三分钟即可到达。

📤 工具返回结果:
   - 类型: <class 'langchain_core.messages.tool.ToolMessage'>
   - content: 从城市 长沙 出发，到目标城市 桂林,使用意念传送，只需要三分钟即可到达。

✅ 工具执行完成，消息历史已更新

📝 当前消息历史长度: 8


In [14]:
# =============================================
# 步骤8: 第二次调用 LLM (生成最终答案)
# =============================================
print("=" * 50)
print("🤖 步骤8: 第二次调用 LLM (整合工具结果)")
print("=" * 50)

print(f"📤 将工具结果发送回 LLM，让其整合答案...")
print(f"📝 消息历史:")
for i, msg in enumerate(messages):
    role = msg.type if hasattr(msg, 'type') else 'human'
    content = msg.content if hasattr(msg, 'content') else str(msg)
    print(f"   [{i}] {role}: {content[:80]}..." if len(content) > 80 else f"   [{i}] {role}: {content}")

print("\n⏳ 等待 LLM 响应...")
final_response = llm_with_tools.invoke(messages)

print(f"\n📥 LLM 最终回答:")
print("-" * 50)
print(final_response.content)
print("-" * 50)

messages.append(final_response)

🤖 步骤8: 第二次调用 LLM (整合工具结果)
📤 将工具结果发送回 LLM，让其整合答案...
📝 消息历史:
   [0] human: ('human', '帮我规划一条从长沙到桂林的自驾路线')
   [1] ai: 
   [2] tool: 从城市 长沙 出发，到目标城市 桂林,使用意念传送，只需要三分钟即可到达。
   [3] tool: 从城市 长沙 出发，到目标城市 桂林,使用意念传送，只需要三分钟即可到达。
   [4] tool: 从城市 长沙 出发，到目标城市 桂林,使用意念传送，只需要三分钟即可到达。
   [5] tool: 从城市 长沙 出发，到目标城市 桂林,使用意念传送，只需要三分钟即可到达。
   [6] tool: 从城市 长沙 出发，到目标城市 桂林,使用意念传送，只需要三分钟即可到达。
   [7] tool: 从城市 长沙 出发，到目标城市 桂林,使用意念传送，只需要三分钟即可到达。

⏳ 等待 LLM 响应...

📥 LLM 最终回答:
--------------------------------------------------
目前意念传送技术尚未实现，现实中从长沙到桂林的自驾路线需要依靠实际交通方式。根据规划，长沙到桂林的自驾距离约为550-600公里，预计耗时约6-7小时，途经京港澳高速、泉南高速等主要高速公路，沿途可经过衡阳、永州等地。

如需详细路线（如出入口、服务区、实时路况或备选路线），请告诉我，我可以进一步协助！
--------------------------------------------------


In [15]:
# =============================================
# 总结: 完整的执行流程
# =============================================
print("=" * 50)
print("📊 执行流程总结")
print("=" * 50)

print("""
完整的 Agent 执行流程:

1️⃣  用户提出问题
    └─> "帮我规划一条从长沙到桂林的自驾路线"

2️⃣  LLM 第一次调用 (判断是否需要工具)
    └─> LLM 分析问题，发现需要调用 get_route_plan 工具

3️⃣  执行工具调用
    └─> 调用 get_route_plan(origin_city="长沙", target_city="桂林")
    └─> 获取工具返回的路线信息

4️⃣  LLM 第二次调用 (整合结果)
    └─> 将工具结果发送给 LLM
    └─> LLM 生成自然语言回答

这就是 ReAct (Reasoning + Acting) 模式!
""")

print(f"📝 最终消息历史长度: {len(messages)}")

📊 执行流程总结

完整的 Agent 执行流程:

1️⃣  用户提出问题
    └─> "帮我规划一条从长沙到桂林的自驾路线"

2️⃣  LLM 第一次调用 (判断是否需要工具)
    └─> LLM 分析问题，发现需要调用 get_route_plan 工具

3️⃣  执行工具调用
    └─> 调用 get_route_plan(origin_city="长沙", target_city="桂林")
    └─> 获取工具返回的路线信息

4️⃣  LLM 第二次调用 (整合结果)
    └─> 将工具结果发送给 LLM
    └─> LLM 生成自然语言回答

这就是 ReAct (Reasoning + Acting) 模式!

📝 最终消息历史长度: 9
